# Comparaison des bibliothèques — SAE 5.03 Datamining

Ce notebook évalue et compare toutes les configurations générées par la pipeline étendue :
- **SW_MODE** : S0 (aucune suppression), S1 (suppression totale), S2 (suppression partielle — préserve négation/intensité)
- **LEMMA_LIB** : spaCy vs Stanza
- **VECT_LIB** : TF-IDF vs BM25

In [ ]:
import glob
import pickle
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score, accuracy_score

In [ ]:
pkl_files = sorted(glob.glob('vectorisation-du-texte/output/*_FINAL.pkl'))
print(f"{len(pkl_files)} fichiers trouvés")

results = []
for path in pkl_files:
    with open(path, 'rb') as f:
        data = pickle.load(f)

    X = data['X_normalized']
    y = data['target']
    cfg = data.get('config', {})
    name = cfg.get('name', path.split('/')[-1].replace('_FINAL.pkl', ''))

    # Extraire sw_mode, lemma_lib, vect_lib depuis le nom si pas dans config
    # Anciens fichiers : config_L1_S1_LEM1_NG3_FINAL → sw_mode déduit de S
    # Nouveaux fichiers : config_L1_S2_LEM1_NG1_SPACY_TFIDF_FINAL
    parts = name.split('_')
    sw_code = int(re.search(r'S(\d)', name).group(1)) if re.search(r'S(\d)', name) else -1
    sw_mode_label = {0: 'S0-none', 1: 'S1-all', 2: 'S2-partial'}.get(sw_code, 'unknown')
    lemma_lib = parts[5] if len(parts) >= 7 else 'SPACY'
    vect_lib = parts[6] if len(parts) >= 7 else 'TFIDF'

    results.append({
        'name': name,
        'path': path,
        'sw_mode': sw_mode_label,
        'lemma_lib': lemma_lib,
        'vect_lib': vect_lib,
        'X': X,
        'y': y,
    })

print(f"Chargement terminé : {len(results)} configurations")

In [ ]:
param_grid = {'C': [0.1, 1, 10], 'penalty': ['l2'], 'solver': ['lbfgs']}
scores = []

for r in results:
    X_tr, X_te, y_tr, y_te = train_test_split(r['X'], r['y'], test_size=0.2, random_state=42)
    grid = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5,
                        scoring='accuracy', n_jobs=-1)
    grid.fit(X_tr, y_tr)
    y_pred = grid.predict(X_te)
    scores.append({
        'config': r['name'],
        'sw_mode': r['sw_mode'],
        'lemma_lib': r['lemma_lib'],
        'vect_lib': r['vect_lib'],
        'accuracy': accuracy_score(y_te, y_pred),
        'f1_macro': f1_score(y_te, y_pred, average='macro'),
        'best_C': grid.best_params_['C'],
    })
    print(f"✓ {r['name']} → acc={scores[-1]['accuracy']:.4f}")

df_scores = pd.DataFrame(scores).sort_values('accuracy', ascending=False)

In [ ]:
display(df_scores[['config', 'sw_mode', 'lemma_lib', 'vect_lib', 'accuracy', 'f1_macro']].head(30))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sw_summary = df_scores.groupby('sw_mode')[['accuracy', 'f1_macro']].mean().reset_index()
sw_summary.plot(kind='bar', x='sw_mode', ax=axes[0], title='Accuracy moyenne par SW_MODE')
sw_summary.plot(kind='bar', x='sw_mode', y='f1_macro', ax=axes[1], title='F1 moyen par SW_MODE')
plt.tight_layout()
plt.savefig('comparaison_sw_mode.png', dpi=150)
plt.show()
print(sw_summary.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df_scores.groupby('lemma_lib')['accuracy'].mean().plot(
    kind='bar', ax=axes[0], title='Accuracy par LEMMA_LIB')
df_scores.groupby('vect_lib')['accuracy'].mean().plot(
    kind='bar', ax=axes[1], title='Accuracy par VECT_LIB')
plt.tight_layout()
plt.savefig('comparaison_libs.png', dpi=150)
plt.show()

In [ ]:
best = df_scores.iloc[0]
print(f"Meilleure configuration : {best['config']}")
print(f"  Accuracy : {best['accuracy']:.4f} | F1 : {best['f1_macro']:.4f}")
print(f"  SW_MODE={best['sw_mode']} | LEMMA={best['lemma_lib']} | VECT={best['vect_lib']}")

# Validation hypothèse négation
s0 = df_scores[df_scores.sw_mode == 'S0-none']['accuracy'].mean()
s1 = df_scores[df_scores.sw_mode == 'S1-all']['accuracy'].mean()
s2 = df_scores[df_scores.sw_mode == 'S2-partial']['accuracy'].mean()
print(f"\nHypothèse négation — accuracy moyenne :")
print(f"  S0 (aucune suppression)  : {s0:.4f}")
print(f"  S1 (suppression totale)  : {s1:.4f}")
print(f"  S2 (suppression partielle): {s2:.4f}")
if s2 > s1:
    print("→ S2 > S1 : conserver les marqueurs de négation AMÉLIORE les performances ✓")
else:
    print("→ S2 ≤ S1 : l'hypothèse de la négation n'est pas confirmée sur ce corpus")